In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier 
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold



# Load dataset

In [20]:
df = pd.read_csv("C:\\Users\\vijay\\OneDrive\\Documents\\Diabetes_Prediction\\Part_1\\cleaned_data.csv")
print(df.head(5))

   NoOfPregency  PlasmaGlucoseCon  BloodPressure  SkinFoldThickness  Insuline  \
0          10.0             108.0           66.0               29.0     130.0   
1           7.0             107.0           74.0               29.0     130.0   
2           0.0             179.0           90.0               27.0     130.0   
3          10.0             125.0           70.0               26.0     130.0   
4          10.0             117.0           72.0               29.0     130.0   

    BMI  DiabetesPedigree   Age            Class  
0  32.4             0.272  42.0  tested_positive  
1  29.6             0.254  31.0  tested_positive  
2  32.0             0.686  23.0  tested_positive  
3  31.1             0.205  41.0  tested_positive  
4  38.0             0.537  34.0  tested_positive  


In [21]:
df.dtypes

NoOfPregency         float64
PlasmaGlucoseCon     float64
BloodPressure        float64
SkinFoldThickness    float64
Insuline             float64
BMI                  float64
DiabetesPedigree     float64
Age                  float64
Class                 object
dtype: object

# Encoding dataset

In [22]:
X = df.drop(columns=['Class'])
X.dtypes

NoOfPregency         float64
PlasmaGlucoseCon     float64
BloodPressure        float64
SkinFoldThickness    float64
Insuline             float64
BMI                  float64
DiabetesPedigree     float64
Age                  float64
dtype: object

In [23]:
y_clf = df["Class"]
y_clf = y_clf.map({'tested_negative': 0, 'tested_positive': 1})
print(y_clf.head())
print(y_clf.dtype)

0    1
1    1
2    1
3    1
4    1
Name: Class, dtype: int64
int64


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test)      

# Decision Tree Baseline

In [25]:
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train_scaled, y_train)
y_pred_dt = dt_clf.predict(X_test_scaled)
y_prob_dt = dt_clf.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))
print("AUC:", roc_auc_score(y_test, y_prob_dt))

[[77 19]
 [22 20]]
              precision    recall  f1-score   support

           0       0.78      0.80      0.79        96
           1       0.51      0.48      0.49        42

    accuracy                           0.70       138
   macro avg       0.65      0.64      0.64       138
weighted avg       0.70      0.70      0.70       138

AUC: 0.6391369047619048


In [27]:
y_pred_dt = dt_clf.predict(X_test_scaled)
y_prob_dt = dt_clf.predict_proba(X_test_scaled)[:, 1]
train_acc= accuracy_score(y_train, dt_clf.predict(X_train_scaled))
test_acc = accuracy_score(y_test, y_pred_dt)

print("Training Accuracy:", round(train_acc, 4))
print("Test Accuracy:", round(test_acc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

print("AUC:", round(roc_auc_score(y_test, y_prob_dt), 4))

Training Accuracy: 1.0
Test Accuracy: 0.7029

Confusion Matrix:
[[77 19]
 [22 20]]

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.80      0.79        96
           1       0.51      0.48      0.49        42

    accuracy                           0.70       138
   macro avg       0.65      0.64      0.64       138
weighted avg       0.70      0.70      0.70       138

AUC: 0.6391


# Controlled Decision Tree:

In [28]:

dt_clf_cont = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_clf_cont.fit(X_train_scaled, y_train)


DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)

In [29]:
y_pred_cont = dt_clf_cont.predict(X_test_scaled)
y_prob_cont = dt_clf_cont.predict_proba(X_test_scaled)[:, 1]
train_acc_cont = accuracy_score(y_train, dt_clf_cont.predict(X_train_scaled))
test_acc_cont = accuracy_score(y_test, y_pred_cont)

print("Training Accuracy:", round(train_acc_cont, 4))
print("Test Accuracy:", round(test_acc_cont, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_cont))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cont))

print("AUC:", round(roc_auc_score(y_test, y_prob_cont), 4))

Training Accuracy: 0.8207
Test Accuracy: 0.7464

Confusion Matrix:
[[80 16]
 [19 23]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.83      0.82        96
           1       0.59      0.55      0.57        42

    accuracy                           0.75       138
   macro avg       0.70      0.69      0.69       138
weighted avg       0.74      0.75      0.74       138

AUC: 0.7907


# Gini vs Entropy comparison

In [ ]:
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_train)
y_pred_gini = dt_gini.predict(X_test_scaled)
gini_acc = accuracy_score(y_test, y_pred_gini)
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_train)
y_pred_entropy = dt_entropy.predict(X_test_scaled)
entropy_acc = accuracy_score(y_test, y_pred_entropy)
print("Gini Accuracy   :", round(gini_acc, 4))
print("Entropy Accuracy:", round(entropy_acc, 4))

Gini Accuracy   : 0.7464
Entropy Accuracy: 0.7609


# Random Forest

In [33]:
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_clf.fit(X_train_scaled, y_train)
y_pred_rf = rf_clf.predict(X_test_scaled)
y_prob_rf = rf_clf.predict_proba(X_test_scaled)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred_rf), 4))
print("Precision:", round(precision_score(y_test, y_pred_rf), 4))
print("Recall   :", round(recall_score(y_test, y_pred_rf), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_rf), 4))
print("AUC      :", round(roc_auc_score(y_test, y_prob_rf), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Accuracy : 0.7681
Precision: 0.6136
Recall   : 0.6429
F1 Score : 0.6279
AUC      : 0.8187

Confusion Matrix:
[[79 17]
 [15 27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.82      0.83        96
           1       0.61      0.64      0.63        42

    accuracy                           0.77       138
   macro avg       0.73      0.73      0.73       138
weighted avg       0.77      0.77      0.77       138



In [34]:
train_acc_rf = accuracy_score(y_train, rf_clf.predict(X_train_scaled))
test_acc_rf = accuracy_score(y_test, y_pred_rf)
print("Training Accuracy:", round(train_acc_rf, 4))
print("Test Accuracy:", round(test_acc_rf, 4))

Training Accuracy: 0.9909
Test Accuracy: 0.7681


In [ ]:
feature_importance = pd.DataFrame({"Feature": X.columns, "Importance": rf_clf.feature_importances_})
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)
print(feature_importance) 

             Feature  Importance
1   PlasmaGlucoseCon    0.278536
5                BMI    0.154500
6   DiabetesPedigree    0.129724
7                Age    0.116892
0       NoOfPregency    0.086112
4           Insuline    0.081448
3  SkinFoldThickness    0.076402
2      BloodPressure    0.076387


# Gradient Boosting

In [36]:
from sklearn.ensemble import GradientBoostingClassifier
# Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_clf.fit(X_train_scaled, y_train)
y_pred_gb = gb_clf.predict(X_test_scaled)
y_prob_gb = gb_clf.predict_proba(X_test_scaled)[:, 1]
print("Accuracy :", round(accuracy_score(y_test, y_pred_gb), 4))
print("Precision:", round(precision_score(y_test, y_pred_gb), 4))
print("Recall   :", round(recall_score(y_test, y_pred_gb), 4))
print("F1 Score :", round(f1_score(y_test, y_pred_gb), 4))
print("AUC      :", round(roc_auc_score(y_test, y_prob_gb), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb))

Accuracy : 0.7681
Precision: 0.6087
Recall   : 0.6667
F1 Score : 0.6364
AUC      : 0.8333

Confusion Matrix:
[[78 18]
 [14 28]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.81      0.83        96
           1       0.61      0.67      0.64        42

    accuracy                           0.77       138
   macro avg       0.73      0.74      0.73       138
weighted avg       0.78      0.77      0.77       138



In [37]:
train_acc_gb = accuracy_score(y_train, gb_clf.predict(X_train_scaled))
test_acc_gb = accuracy_score(y_test, y_pred_gb)
print("Training Accuracy:", round(train_acc_gb, 4))
print("Test Accuracy:", round(test_acc_gb, 4))

Training Accuracy: 0.9203
Test Accuracy: 0.7681


In [38]:
importance_df = pd.DataFrame({"Feature": X.columns, "Importance": gb_clf.feature_importances_})
importance_df = importance_df.sort_values(by="Importance", ascending=False)
print(importance_df)

             Feature  Importance
1   PlasmaGlucoseCon    0.388588
5                BMI    0.147349
6   DiabetesPedigree    0.135015
7                Age    0.111535
0       NoOfPregency    0.069224
4           Insuline    0.052182
3  SkinFoldThickness    0.049582
2      BloodPressure    0.046525


# Feature ablation study

In [ ]:
feature_importance = pd.DataFrame({"Feature": X.columns, "Importance": rf_clf.feature_importances_}).sort_values(by="Importance", ascending=True)
lowest_5 = feature_importance.head(5)["Feature"].tolist()
keep_features = [col for col in X.columns
    if col not in lowest_5
]
keep_idx = [list(X.columns).index(col) for col in keep_features]
X_train_reduced = X_train_scaled[:, keep_idx]
X_test_reduced = X_test_scaled[:, keep_idx]
rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_train)
y_prob_full = rf_clf.predict_proba(X_test_scaled)[:, 1]
auc_full = roc_auc_score(y_test, y_prob_full)
y_prob_reduced = rf_reduced.predict_proba(X_test_reduced)[:, 1]
auc_reduced = roc_auc_score(y_test, y_prob_reduced)
print("Five Lowest-Importance Features Removed:")
print(lowest_5)
print("\nRemaining Features:")
print(keep_features)
print(f"\nFull Model AUC   : {auc_full:.4f}")
print(f"Reduced Model AUC: {auc_reduced:.4f}")

Five Lowest-Importance Features Removed:
['BloodPressure', 'SkinFoldThickness', 'Insuline', 'NoOfPregency', 'Age']

Remaining Features:
['PlasmaGlucoseCon', 'BMI', 'DiabetesPedigree']

Full Model AUC   : 0.8187
Reduced Model AUC: 0.7755


# Cross-validated comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {"Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
          "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
          "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
          "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)}

results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    results.append({
        "Model": name,
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std()
    })
results_df = pd.DataFrame(results)
print(results_df.round(4))

                         Model  Mean ROC-AUC  Std ROC-AUC
0          Logistic Regression        0.8178       0.0392
1  Decision Tree (max_depth=5)        0.7423       0.0271
2                Random Forest        0.8030       0.0399
3            Gradient Boosting        0.7914       0.0360


# Hyperparameter tuning with GridSearchCV

In [ ]:

pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), RandomForestClassifier(random_state=42))

param_grid = {'randomforestclassifier__n_estimators': [50, 100, 200],
              'randomforestclassifier__max_depth': [5, 10, None],
              'randomforestclassifier__min_samples_leaf': [1, 5]}

cv = StratifiedKFold( n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)

grid_search.fit(X_train, y_train)
print("Best Parameters:")
print(grid_search.best_params_)
print("\nBest CV ROC-AUC:")
print(round(grid_search.best_score_, 4))

Best Parameters:
{'randomforestclassifier__max_depth': 5, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 200}

Best CV ROC-AUC:
0.8188


In [42]:
grid_search.best_params_
grid_search.best_score_

np.float64(0.8187515090543259)

# Manual learning curve

In [ ]:
best_pipeline = grid_search.best_estimator_
best_pipeline = grid_search.best_estimator_
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []
for f in fractions:

    n = int(f * len(X_train))

    X_subset = X_train.iloc[:n]
    y_subset = y_train.iloc[:n]

    best_pipeline.fit(X_subset, y_subset)

    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]
    train_auc = roc_auc_score(y_subset, train_prob)

    test_prob = best_pipeline.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_prob)

    results.append([
        f,
        train_auc,
        test_auc
    ])

learning_curve_df = pd.DataFrame(results, columns=["Training Fraction", "Training AUC", "Test AUC"])

print(learning_curve_df.round(4))

   Training Fraction  Training AUC  Test AUC
0                0.2        0.9815    0.8336
1                0.4        0.9450    0.8475
2                0.6        0.9459    0.8385
3                0.8        0.9313    0.8480
4                1.0        0.9166    0.8447


# Serialize the best model

In [ ]:
joblib.dump(best_pipeline, "best_model.pkl")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
loaded_model = joblib.load("best_model.pkl")
new_patients = pd.DataFrame([
    {
        "NoOfPregency": 2,
        "PlasmaGlucoseCon": 120,
        "BloodPressure": 70,
        "SkinFoldThickness": 25,
        "Insuline": 90,
        "BMI": 28.5,
        "DiabetesPedigree": 0.45,
        "Age": 32
    },
    {
        "NoOfPregency": 8,
        "PlasmaGlucoseCon": 180,
        "BloodPressure": 88,
        "SkinFoldThickness": 35,
        "Insuline": 250,
        "BMI": 42.0,
        "DiabetesPedigree": 1.20,
        "Age": 55
    }
])

predictions = loaded_model.predict(new_patients)
probabilities = loaded_model.predict_proba(new_patients)
print("Predicted Classes:")
print(predictions)
print("\nPredicted Probabilities:")
print(probabilities)

Predicted Classes:
[0 1]

Predicted Probabilities:
[[0.68970115 0.31029885]
 [0.17122358 0.82877642]]
